In [1]:
import pypdf
import langchain_community
import langchain_text_splitters
import sentence_transformers
import faiss
import langchain_groq
import dotenv

print("All required libraries imported successfully!")

C:\Users\chirag gupta\AppData\Local\Temp\ipykernel_17004\1727986457.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  import langchain_community
d:\RAG_project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


All required libraries imported successfully!


In [2]:
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader


def process_all_pdfs(pdf_directory):
    """Load all PDF files from a directory and its subdirectories."""

    all_documents = []
    pdf_dir = Path(pdf_directory)

    # Check whether the directory exists
    if not pdf_dir.is_dir():
        raise NotADirectoryError(
            f"PDF directory not found: {pdf_dir.resolve()}"
        )

    # Find all PDF files recursively
    pdf_files = sorted(pdf_dir.rglob("*.pdf"))

    print(f"Found {len(pdf_files)} PDF files to process")

    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")

        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            # Add source information to each page's metadata
            for doc in documents:
                doc.metadata["source_file"] = pdf_file.name
                doc.metadata["file_type"] = "pdf"

            all_documents.extend(documents)

            print(f"  Loaded {len(documents)} pages")

        except Exception as e:
            print(f"  Error loading {pdf_file.name}: {e}")

    print(f"\nTotal PDF pages loaded: {len(all_documents)}")

    return all_documents


# Load all PDFs from the data directory
all_pdf_documents = process_all_pdfs("../data")

Found 1 PDF files to process

Processing: book1.pdf


fontTools is required to fully parse the encoding of a CFF Type1 font in font dictionary {'/Type': '/Font', '/Subtype': '/Type1', '/FirstChar': 32, '/LastChar': 240, '/Widths': [259, 352, 426, 519, 519, 759, 870, 185, 389, 389, 500, 519, 259, 333, 259, 278, 519, 519, 519, 519, 519, 519, 519, 519, 519, 519, 259, 259, 519, 519, 519, 444, 746, 648, 549, 685, 704, 593, 574, 722, 722, 333, 333, 667, 556, 852, 741, 741, 556, 741, 611, 478, 611, 704, 648, 963, 648, 574, 593, 389, 278, 389, 519, 500, 278, 444, 519, 444, 537, 444, 296, 481, 556, 278, 278, 537, 278, 815, 556, 519, 537, 519, 370, 389, 315, 556, 481, 722, 519, 481, 444, 389, 222, 389, 519, 259, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 519, 519, 0, 0, 0, 0, 0, 800, 990, 0, 0, 259, 0, 0, 259, 519, 259, 259, 0, 556, 259, 259, 259, 259, 259, 0, 0, 259, 0, 0, 0, 0, 0, 259, 0, 259, 259, 0, 0, 0, 259, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 185, 0, 259, 0, 0, 0, 0, 0, 0, 556, 0, 0, 0, 0,

  Loaded 105 pages

Total PDF pages loaded: 105


In [3]:
print("Total page documents:", len(all_pdf_documents))

if all_pdf_documents:
    print("\nFirst page metadata:")
    print(all_pdf_documents[0].metadata)

    print("\nFirst page text:")
    print(all_pdf_documents[0].page_content[:1000])

    empty_pages = sum(
        1 for doc in all_pdf_documents
        if not doc.page_content.strip()
    )

    print("\nPages with no extracted text:", empty_pages)

Total page documents: 105

First page metadata:
{'producer': 'BookVirtual Corp. Patents Pending.', 'creator': 'BookVirtual Digital Works', 'creationdate': '2000', 'keywords': "Carroll, Alice, Wonderland, children's, 1865; v.1.2", 'title': "Alice's Adventures in Wonderland", 'moddate': '2000-11-27T16:31:36-08:00', 'subject': "children's literature", 'author': 'Lewis Carroll; BkV0000010; Bkslr0000001; ISBN<n/a>;', 'source': '..\\data\\book1.pdf', 'total_pages': 105, 'page': 0, 'page_label': '1', 'source_file': 'book1.pdf', 'file_type': 'pdf'}

First page text:
BY LEWIS CARROLL ILLUSTRATED BY JOHN TENNIEL
NAVIGATE
A LICE ’S
Adventures in W onderland
CONTROL
CLOSE THE BOOK
TURN THE PAGE
The world’ s
most precise
replica
of the world’ s 
most famous
children’ s book!
In 1998, Peter Zelchenko
began a project for Volume-
One Publishing: to create an
exact digital replica of Lewis
Carroll’s ﬁrst edition of Alice.
Working with the original
1865 edition and numerous
other editions at the Newberr

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Create the text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

# Split all 105 pages into smaller chunks
chunks = text_splitter.split_documents(all_pdf_documents)

print("Total pages:", len(all_pdf_documents))
print("Total chunks:", len(chunks))
print("\nFirst chunk:")
print(chunks[0].page_content)
print("\nFirst chunk metadata:")
print(chunks[0].metadata)

Total pages: 105
Total chunks: 245

First chunk:
BY LEWIS CARROLL ILLUSTRATED BY JOHN TENNIEL
NAVIGATE
A LICE ’S
Adventures in W onderland
CONTROL
CLOSE THE BOOK
TURN THE PAGE
The world’ s
most precise
replica
of the world’ s 
most famous
children’ s book!
In 1998, Peter Zelchenko
began a project for Volume-
One Publishing: to create an
exact digital replica of Lewis
Carroll’s ﬁrst edition of Alice.
Working with the original
1865 edition and numerous
other editions at the Newberry
Library in Chicago, Zelchenko
created a digital masterpiece in
his own right, a testament to
the original work of Lewis
Carroll (aka Prof. Charles
Dodgson) who personally
directed the typography for the
ﬁrst Alice.
After much analyis, Peter then
painstakingly matched letter to
letter, line to line, of his new
digital edition to that of the
original. After weeks of toil he
created an exact replica of the
original! The book was added
to VolumeOne’s print-on-
demand offering. While a PDF
version is offered on va

In [5]:
from sentence_transformers import SentenceTransformer

# Load the embedding model
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

# Extract text from every chunk
chunk_texts = [chunk.page_content for chunk in chunks]

# Convert chunk text into numerical vectors
embeddings = embedding_model.encode(
    chunk_texts,
    show_progress_bar=True,
    convert_to_numpy=True
)

print("Number of chunks:", len(chunks))
print("Embedding shape:", embeddings.shape)
print("First embedding:", embeddings[0][:10])

Batches: 100%|██████████| 8/8 [00:20<00:00,  2.52s/it]

Number of chunks: 245
Embedding shape: (245, 384)
First embedding: [-0.03045103  0.00500179 -0.04448766 -0.00563105 -0.05024881 -0.01978334
 -0.01706642  0.01490893  0.07596821  0.09336282]


In [6]:

import numpy as np

# Convert embeddings to float32, as required by FAISS
embeddings = np.asarray(embeddings, dtype="float32")

# Create a FAISS index for 384-dimensional vectors
dimension = embeddings.shape[1]
faiss_index = faiss.IndexFlatL2(dimension)

# Store all chunk embeddings in the index
faiss_index.add(embeddings)

print("Vector dimensions:", faiss_index.d)
print("Vectors stored:", faiss_index.ntotal)

Vector dimensions: 384
Vectors stored: 245


In [7]:
# Select the first chunk
chunk = chunks[0]

# View its text
print("CHUNK TEXT:")
print(chunk.page_content)

# View its metadata
print("\nCHUNK METADATA:")
print(chunk.metadata)

CHUNK TEXT:
BY LEWIS CARROLL ILLUSTRATED BY JOHN TENNIEL
NAVIGATE
A LICE ’S
Adventures in W onderland
CONTROL
CLOSE THE BOOK
TURN THE PAGE
The world’ s
most precise
replica
of the world’ s 
most famous
children’ s book!
In 1998, Peter Zelchenko
began a project for Volume-
One Publishing: to create an
exact digital replica of Lewis
Carroll’s ﬁrst edition of Alice.
Working with the original
1865 edition and numerous
other editions at the Newberry
Library in Chicago, Zelchenko
created a digital masterpiece in
his own right, a testament to
the original work of Lewis
Carroll (aka Prof. Charles
Dodgson) who personally
directed the typography for the
ﬁrst Alice.
After much analyis, Peter then
painstakingly matched letter to
letter, line to line, of his new
digital edition to that of the
original. After weeks of toil he
created an exact replica of the
original! The book was added
to VolumeOne’s print-on-
demand offering. While a PDF
version is offered on various
portals of the Net, BookVirtual

In [9]:
import pandas as pd

chunk_data = []

for i, chunk in enumerate(chunks):
    chunk_data.append({
        "chunk_id": i,
        "text_preview": chunk.page_content[:150],
        "source_file": chunk.metadata.get("source_file"),
        "page": chunk.metadata.get("page"),
        "title": chunk.metadata.get("title"),
        "author": chunk.metadata.get("author")
    })

chunks_df = pd.DataFrame(chunk_data)

display(chunks_df.head(10))

,chunk_id,text_preview,source_file,page,title,author
0,0,BY LEWIS CARROLL ILLUSTRATED BY JOHN TENNIEL\n...,book1.pdf,0,Alice's Adventures in Wonderland,Lewis Carroll; BkV0000010; Bkslr0000001; ISBN<...
1,1,created an exact replica of the\noriginal! The...,book1.pdf,0,Alice's Adventures in Wonderland,Lewis Carroll; BkV0000010; Bkslr0000001; ISBN<...
2,2,RABBIT-HOLE. 1,book1.pdf,1,Alice's Adventures in Wonderland,Lewis Carroll; BkV0000010; Bkslr0000001; ISBN<...
3,3,Digital Interface by BookVirtual Corp. U.S. Pa...,book1.pdf,2,Alice's Adventures in Wonderland,Lewis Carroll; BkV0000010; Bkslr0000001; ISBN<...
4,4,Navigate Control Internet\nDigital Interface b...,book1.pdf,3,Alice's Adventures in Wonderland,Lewis Carroll; BkV0000010; Bkslr0000001; ISBN<...
5,5,Navigate Control Internet\nDigital Interface b...,book1.pdf,4,Alice's Adventures in Wonderland,Lewis Carroll; BkV0000010; Bkslr0000001; ISBN<...
6,6,(312) 733-2473\nThe text of this book was orig...,book1.pdf,4,Alice's Adventures in Wonderland,Lewis Carroll; BkV0000010; Bkslr0000001; ISBN<...
7,7,Navigate Control Internet\nDigital Interface b...,book1.pdf,5,Alice's Adventures in Wonderland,Lewis Carroll; BkV0000010; Bkslr0000001; ISBN<...
8,8,"Of wonders wild and new,\nIn friendly chat wit...",book1.pdf,5,Alice's Adventures in Wonderland,Lewis Carroll; BkV0000010; Bkslr0000001; ISBN<...
9,9,Digital Interface by BookVirtual Corp. U.S. Pa...,book1.pdf,6,Alice's Adventures in Wonderland,Lewis Carroll; BkV0000010; Bkslr0000001; ISBN<...


In [10]:
# Convert the question into an embedding
query = "What happens when Alice follows the White Rabbit?"

query_embedding = embedding_model.encode(
    [query],
    convert_to_numpy=True
).astype("float32")

# Search FAISS for the 5 nearest vectors
distances, indices = faiss_index.search(query_embedding, k=5)

# Display the retrieved chunks
for rank, (index, distance) in enumerate(
    zip(indices[0], distances[0]), start=1
):
    chunk = chunks[index]

    print(f"\n--- Result {rank} ---")
    print("Distance:", distance)
    print("Source:", chunk.metadata.get("source_file"))
    print("Page:", chunk.metadata.get("page_label"))
    print("Text:", chunk.page_content[:500])


--- Result 1 ---
Distance: 0.58933836
Source: book1.pdf
Page: 9
Text: curred to her that she ought to have wondered
at this, but at the time it all seemed quite
natural) ; but when the Rabbit actually took a
watch out of its waistcoat-pocket, and looked at
it, and then hurried on, Alice started to her
feet, for it ﬂashed across her mind that she had
never before seen a rabbit with either a waist-
coat-pocket or a watch to take out of it, and,
DOWN THE2 RABBIT-HOLE. 3
Fit Page Full Screen On/Off Close Book
B 2

--- Result 2 ---
Distance: 0.5931331
Source: book1.pdf
Page: 16
Text: deep and reaching half down the hall.
After a time she heard a little pattering of
feet in the distance, and she hastily dried her
eyes to see what was coming. It was the White
Rabbit returning, splendidly dressed, with a pair
of white kid gloves in one hand and a large
fan in the other : he came trotting along in a
great hurry, muttering to himself as he came,
“Oh ! the Duchess, the Duchess ! Oh ! won’t she
b

In [12]:
query = "Why does Alice follow the White Rabbit?"
query_embedding = embedding_model.encode(
    [query],
    convert_to_numpy=True
).astype("float32")

# Search FAISS for the 5 nearest vectors
distances, indices = faiss_index.search(query_embedding, k=5)

# Display the retrieved chunks
for rank, (index, distance) in enumerate(
    zip(indices[0], distances[0]), start=1
):
    chunk = chunks[index]

    print(f"\n--- Result {rank} ---")
    print("Distance:", distance)
    print("Source:", chunk.metadata.get("source_file"))
    print("Page:", chunk.metadata.get("page_label"))
    print("Text:", chunk.page_content[:500])


--- Result 1 ---
Distance: 0.6353041
Source: book1.pdf
Page: 16
Text: deep and reaching half down the hall.
After a time she heard a little pattering of
feet in the distance, and she hastily dried her
eyes to see what was coming. It was the White
Rabbit returning, splendidly dressed, with a pair
of white kid gloves in one hand and a large
fan in the other : he came trotting along in a
great hurry, muttering to himself as he came,
“Oh ! the Duchess, the Duchess ! Oh ! won’t she
be savage if I ’ve kept her waiting !” Alice
felt so desperate that she was ready to ask

--- Result 2 ---
Distance: 0.65251225
Source: book1.pdf
Page: 9
Text: curred to her that she ought to have wondered
at this, but at the time it all seemed quite
natural) ; but when the Rabbit actually took a
watch out of its waistcoat-pocket, and looked at
it, and then hurried on, Alice started to her
feet, for it ﬂashed across her mind that she had
never before seen a rabbit with either a waist-
coat-pocket or a watch to t

In [13]:

import numpy as np

def mmr_search(query, k=5, fetch_k=15, lambda_mult=0.5):
    # Convert the question into an embedding
    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True
    ).astype("float32")[0]

    # Retrieve more candidates than we ultimately need
    distances, indices = faiss_index.search(
        query_embedding.reshape(1, -1),
        fetch_k
    )

    candidate_indices = [
        int(i) for i in indices[0] if i >= 0
    ]

    # Normalize vectors so dot product represents cosine similarity
    candidate_vectors = embeddings[candidate_indices]
    candidate_vectors = candidate_vectors / (
        np.linalg.norm(candidate_vectors, axis=1, keepdims=True) + 1e-12
    )

    query_vector = query_embedding / (
        np.linalg.norm(query_embedding) + 1e-12
    )

    selected = []
    remaining = list(range(len(candidate_indices)))

    while remaining and len(selected) < k:
        best_position = None
        best_score = -float("inf")

        for position in remaining:
            relevance = np.dot(
                candidate_vectors[position], query_vector
            )

            if selected:
                redundancy = max(
                    np.dot(
                        candidate_vectors[position],
                        candidate_vectors[chosen]
                    )
                    for chosen in selected
                )
            else:
                redundancy = 0.0

            score = (
                lambda_mult * relevance
                - (1 - lambda_mult) * redundancy
            )

            if score > best_score:
                best_score = score
                best_position = position

        selected.append(best_position)
        remaining.remove(best_position)

    return [
        chunks[candidate_indices[position]]
        for position in selected
    ]


In [14]:

query = "Why does Alice follow the White Rabbit?"

mmr_results = mmr_search(
    query,
    k=5,
    fetch_k=15,
    lambda_mult=0.5
)

for rank, chunk in enumerate(mmr_results, start=1):
    print(f"\n--- MMR Result {rank} ---")
    print("Source:", chunk.metadata.get("source_file"))
    print("Page:", chunk.metadata.get("page_label"))
    print("Text:", chunk.page_content[:400])



--- MMR Result 1 ---
Source: book1.pdf
Page: 16
Text: deep and reaching half down the hall.
After a time she heard a little pattering of
feet in the distance, and she hastily dried her
eyes to see what was coming. It was the White
Rabbit returning, splendidly dressed, with a pair
of white kid gloves in one hand and a large
fan in the other : he came trotting along in a
great hurry, muttering to himself as he came,
“Oh ! the Duchess, the Duchess ! Oh 

--- MMR Result 2 ---
Source: book1.pdf
Page: 9
Text: curred to her that she ought to have wondered
at this, but at the time it all seemed quite
natural) ; but when the Rabbit actually took a
watch out of its waistcoat-pocket, and looked at
it, and then hurried on, Alice started to her
feet, for it ﬂashed across her mind that she had
never before seen a rabbit with either a waist-
coat-pocket or a watch to take out of it, and,
DOWN THE2 RABBIT-HOLE. 

--- MMR Result 3 ---
Source: book1.pdf
Page: 86
Text: So Alice began telling them her ad

In [15]:

query = "Why does Alice follow the White Rabbit?"

# Semantic search: retrieve the top 5 nearest vectors
query_embedding = embedding_model.encode(
    [query],
    convert_to_numpy=True
).astype("float32")

distances, indices = faiss_index.search(query_embedding, 5)

semantic_results = [
    chunks[int(i)] for i in indices[0] if i >= 0
]

# MMR search
mmr_results = mmr_search(
    query,
    k=5,
    fetch_k=15,
    lambda_mult=0.5
)

# Compare the results
for method, results in [
    ("Semantic Search", semantic_results),
    ("MMR", mmr_results)
]:
    print(f"\n{'=' * 20} {method} {'=' * 20}")

    for rank, chunk in enumerate(results, start=1):
        print(
            f"\nRank {rank} | "
            f"Page {chunk.metadata.get('page_label')}"
        )
        print(chunk.page_content[:250].replace("\n", " "))



==================== Semantic Search ====================

Rank 1 | Page 16
deep and reaching half down the hall. After a time she heard a little pattering of feet in the distance, and she hastily dried her eyes to see what was coming. It was the White Rabbit returning, splendidly dressed, with a pair of white kid gloves in 

Rank 2 | Page 9
curred to her that she ought to have wondered at this, but at the time it all seemed quite natural) ; but when the Rabbit actually took a watch out of its waistcoat-pocket, and looked at it, and then hurried on, Alice started to her feet, for it ﬂash

Rank 3 | Page 11
Alice was not a bit hurt, and she jumped up on to her feet in a moment : she looked up, but it was all dark overhead ; before her was another long passage, and the White Rabbit was still in sight, hurrying down it. There was not a moment to be lost :

Rank 4 | Page 9
then she looked at the sides of the well, and noticed that they were filled with cupboards and bookshelves : here and 

In [16]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq

load_dotenv(override=True)

groq_api_key = os.getenv("GROQ_API_KEY")

if not groq_api_key:
    raise ValueError(
        "GROQ_API_KEY not found. Check your .env file."
    )

llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0,
    max_tokens=1024
)

response = llm.invoke(
    "Reply with exactly: Groq connection successful"
)

print(response.content)

Groq connection successful


In [17]:

from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import LLMChainExtractor
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

# Reuse your existing embedding model through LangChain
embedding_function = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# Create a LangChain FAISS vector store from existing vectors and chunks
vector_store = FAISS(
    embedding_function=embedding_function,
    index=faiss_index,
    docstore=__import__(
        "langchain_community.docstore.in_memory",
        fromlist=["InMemoryDocstore"]
    ).InMemoryDocstore({
        str(i): chunk for i, chunk in enumerate(chunks)
    }),
    index_to_docstore_id={
        i: str(i) for i in range(len(chunks))
    }
)

# Retrieve the 5 most relevant chunks
base_retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5}
)

# Use Groq to extract question-relevant text
compressor = LLMChainExtractor.from_llm(llm)

# Combine retrieval and compression
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=base_retriever
)

print("Contextual compression retriever created!")


C:\Users\chirag gupta\AppData\Local\Temp\ipykernel_17004\2560543948.py:8: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_function = HuggingFaceEmbeddings(
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 325.73it/s]


Contextual compression retriever created!


In [18]:

query = "Why does Alice follow the White Rabbit?"

# Retrieve and compress the relevant chunks
compressed_results = compression_retriever.invoke(query)

print("Number of compressed chunks:", len(compressed_results))

for rank, chunk in enumerate(compressed_results, start=1):
    print(f"\n--- Compressed Result {rank} ---")
    print("Source:", chunk.metadata.get("source_file"))
    print("Page:", chunk.metadata.get("page_label"))
    print("Text:", chunk.page_content)


Number of compressed chunks: 4

--- Compressed Result 1 ---
Source: book1.pdf
Page: 16
Text: After a time she heard a little pattering of
feet in the distance, and she hastily dried her
eyes to see what was coming. It was the White
Rabbit returning, splendidly dressed, with a pair
of white kid gloves in one hand and a large
fan in the other : he came trotting along in a
great hurry, muttering to himself as he came,
“Oh ! the Duchess, the Duchess ! Oh ! won’t she
be savage if I ’ve kept her waiting !” Alice
felt so desperate that she was ready to ask help

--- Compressed Result 2 ---
Source: book1.pdf
Page: 9
Text: when the Rabbit actually took a watch out of its waistcoat‑pocket, and looked at it, and then hurried on, Alice started to her feet, for it ﬂashed across her mind that she had never before seen a rabbit with either a waist‑coat‑pocket or a watch to take out of it

--- Compressed Result 3 ---
Source: book1.pdf
Page: 11
Text: Alice was not a bit hurt, and she jumped up
on to he

In [19]:

query = "Why does Alice follow the White Rabbit?"

# Semantic search
query_embedding = embedding_model.encode(
    [query],
    convert_to_numpy=True
).astype("float32")

_, indices = faiss_index.search(query_embedding, 5)
semantic_results = [
    chunks[int(i)] for i in indices[0] if i >= 0
]

# MMR
mmr_results = mmr_search(
    query, k=5, fetch_k=15, lambda_mult=0.5
)

# Contextual compression
compressed_results = compression_retriever.invoke(query)

# Prepare comparison table
rows = []

for method, results in [
    ("Semantic Search", semantic_results),
    ("MMR", mmr_results),
    ("Contextual Compression", compressed_results)
]:
    for rank, chunk in enumerate(results, start=1):
        rows.append({
            "method": method,
            "rank": rank,
            "page": chunk.metadata.get("page_label"),
            "text": chunk.page_content[:250].replace("\n", " ")
        })

comparison_df = pd.DataFrame(rows)
display(comparison_df)


,method,rank,page,text
0,Semantic Search,1,16,deep and reaching half down the hall. After a ...
1,Semantic Search,2,9,curred to her that she ought to have wondered ...
2,Semantic Search,3,11,"Alice was not a bit hurt, and she jumped up on..."
3,Semantic Search,4,9,"then she looked at the sides of the well, and ..."
4,Semantic Search,5,86,So Alice began telling them her adventures fro...
5,MMR,1,16,deep and reaching half down the hall. After a ...
6,MMR,2,9,curred to her that she ought to have wondered ...
7,MMR,3,86,So Alice began telling them her adventures fro...
8,MMR,4,34,"“So you did, old fellow !” said the others. “W..."
9,MMR,5,16,"Hearthrug, near the Fender. (with Alice’s love..."


In [67]:

def answer_question(query):
    # Step 1: Retrieve the 5 most relevant chunks
    retrieved_chunks = base_retriever.invoke(query)

    # Step 2: Combine the retrieved text into context
    context = "\n\n".join(
        f"Source: {chunk.metadata.get('source_file')}, "
        f"Page: {chunk.metadata.get('page_label')}\n"
        f"{chunk.page_content}"
        for chunk in retrieved_chunks
    )

    # Step 3: Create the prompt
    prompt = f"""
You are a helpful assistant answering questions about a book.

Answer the question using only the context provided below.
If the answer is not available in the context, say:
"I could not find the answer in the retrieved book passages."

Do not include page numbers or citations in the answer.

Context:
{context}

Question:
{query}
"""

    # Step 4: Ask Groq to generate the answer
    response = llm.invoke(prompt)

    # Step 5: Return the answer and retrieved sources
    return {
        "question": query,
        "answer": response.content,
        "sources": [
            {
                "source_file": chunk.metadata.get("source_file"),
                "page": chunk.metadata.get("page_label")
            }
            for chunk in retrieved_chunks
        ]
    }


In [69]:
result = answer_question(
    "Why does Alice follow the White Rabbit?"
)

print("QUESTION:")
print(result["question"])

print("\nANSWER:")
print(result["answer"])

print("\nSOURCES:")
for source in result["sources"]:
    print(f"Source: {source['source_file']}, Page: {source['page']}")

QUESTION:
Why does Alice follow the White Rabbit?

ANSWER:
Alice follows the White Rabbit because she is intrigued by its strange, hurried behavior—seeing a rabbit with a waist‑coat pocket and a watch, hearing it mutter that it’s late, and feeling a sudden, desperate curiosity that makes her run after it to see where it’s going.

SOURCES:
Source: book1.pdf, Page: 16
Source: book1.pdf, Page: 9
Source: book1.pdf, Page: 11
Source: book1.pdf, Page: 9
Source: book1.pdf, Page: 86
